In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)

sns.set_theme(style="whitegrid")

In [2]:
clean_path = Path(
    "../05_Clean_Data/marketing_data_clean.csv"
)

rfm_path = Path(
    "../05_Clean_Data/marketing_data_rfm.csv"
)

df = pd.read_csv(clean_path)

df_rfm = pd.read_csv(rfm_path)

print("Clean Dataset:", df.shape)
print("RFM Dataset:", df_rfm.shape)

Clean Dataset: (2240, 40)
RFM Dataset: (2240, 46)


In [3]:
clean_path = Path(
    "../05_Clean_Data/marketing_data_clean.csv"
)

rfm_path = Path(
    "../05_Clean_Data/marketing_data_rfm.csv"
)

df = pd.read_csv(clean_path)

df_rfm = pd.read_csv(rfm_path)

print("Clean Dataset:", df.shape)
print("RFM Dataset:", df_rfm.shape)

Clean Dataset: (2240, 40)
RFM Dataset: (2240, 46)


In [4]:
total_customers = df["ID"].nunique()

total_spending = df["Total_Spending"].sum()

total_purchases = df["Total_Purchases"].sum()

avg_spending = df["Total_Spending"].mean()

avg_purchases = df["Total_Purchases"].mean()

campaign_response_rate = (
    df["Response"].mean() * 100
)

flagged_customers = (
    df["Has_Data_Quality_Issue"].sum()
)

In [5]:
executive_kpis = pd.DataFrame({
    "KPI": [
        "Total Customers",
        "Total Spending",
        "Total Purchases",
        "Average Spending per Customer",
        "Average Purchases per Customer",
        "Last Campaign Response Rate (%)",
        "Flagged Customers"
    ],
    "Value": [
        total_customers,
        total_spending,
        total_purchases,
        avg_spending,
        avg_purchases,
        campaign_response_rate,
        flagged_customers
    ]
})

display(executive_kpis.round(2))

,KPI,Value
0,Total Customers,2240.00
1,Total Spending,1356988.00
2,Total Purchases,28083.00
3,Average Spending per Customer,605.80
4,Average Purchases per Customer,12.54
5,Last Campaign Response Rate (%),14.91
6,Flagged Customers,7.00


In [6]:
age_summary = (
    df.groupby("Age_Group", observed=True)
    .agg(
        Customers=("ID", "nunique"),
        Total_Spending=("Total_Spending", "sum"),
        Avg_Spending=("Total_Spending", "mean"),
        Avg_Purchases=("Total_Purchases", "mean")
    )
    .reset_index()
)

display(age_summary.round(2))

,Age_Group,Customers,Total_Spending,Avg_Spending,Avg_Purchases
0,18-24,61,49445,810.57,12.95
1,25-34,363,193207,532.25,10.85
2,35-44,740,387185,523.22,11.55
3,45-54,506,315446,623.41,13.18
4,55-64,460,309865,673.62,13.82
5,65+,107,99900,933.64,16.48


In [7]:
top_age_by_customers = age_summary.loc[
    age_summary["Customers"].idxmax()
]

print(
    "Largest Age Group:",
    top_age_by_customers["Age_Group"]
)

Largest Age Group: 35-44


In [8]:
top_age_by_spending = age_summary.loc[
    age_summary["Avg_Spending"].idxmax()
]

print(
    "Highest Average Spending Age Group:",
    top_age_by_spending["Age_Group"]
)

Highest Average Spending Age Group: 65+


In [10]:
income_summary = (
    df.groupby("Income_Group", observed=True)
    .agg(
        Customers=("ID", "nunique"),
        Avg_Income=("Income", "mean"),
        Avg_Spending=("Total_Spending", "mean"),
        Avg_Purchases=("Total_Purchases", "mean"),
        Response_Rate=("Response", "mean")
    )
    .reset_index()
)

income_summary["Response_Rate"] *= 100

display(income_summary.round(2))

,Income_Group,Customers,Avg_Income,Avg_Spending,Avg_Purchases,Response_Rate
0,High Income,554,80580.90,1357.65,19.32,26.90
1,Low Income,554,25187.11,81.59,5.45,10.47
2,Lower-Middle Income,554,43060.12,231.32,8.77,12.45
3,Upper-Middle Income,554,60160.88,757.74,16.68,10.29


In [11]:
product_columns = {
    "MntWines": "Wines",
    "MntFruits": "Fruits",
    "MntMeatProducts": "Meat Products",
    "MntFishProducts": "Fish Products",
    "MntSweetProducts": "Sweet Products",
    "MntGoldProds": "Gold Products"
}

product_summary = pd.DataFrame({
    "Product_Category": list(product_columns.values()),
    "Total_Spending": [
        df[col].sum()
        for col in product_columns
    ],
    "Customers": [
        (df[col] > 0).sum()
        for col in product_columns
    ]
})

product_summary["Spending_Share"] = (
    product_summary["Total_Spending"]
    / product_summary["Total_Spending"].sum()
    * 100
)

product_summary["Avg_Spending_Per_Buyer"] = (
    product_summary["Total_Spending"]
    / product_summary["Customers"].replace(0, np.nan)
)

product_summary = product_summary.sort_values(
    "Total_Spending",
    ascending=False
)

display(product_summary.round(2))

,Product_Category,Total_Spending,Customers,Spending_Share,Avg_Spending_Per_Buyer
0,Wines,680816,2227,50.17,305.71
2,Meat Products,373968,2239,27.56,167.02
5,Gold Products,98609,2179,7.27,45.25
3,Fish Products,84057,1856,6.19,45.29
4,Sweet Products,60621,1821,4.47,33.29
1,Fruits,58917,1840,4.34,32.02


In [12]:
top_product = product_summary.iloc[0]

print(
    "Top Product Category:",
    top_product["Product_Category"]
)

print(
    "Spending Share:",
    round(top_product["Spending_Share"], 2),
    "%"
)

Top Product Category: Wines
Spending Share: 50.17 %


In [13]:
channel_columns = {
    "NumWebPurchases": "Web",
    "NumCatalogPurchases": "Catalog",
    "NumStorePurchases": "Store"
}

channel_summary = pd.DataFrame({
    "Channel": list(channel_columns.values()),
    "Total_Purchases": [
        df[col].sum()
        for col in channel_columns
    ],
    "Channel_Buyers": [
        (df[col] > 0).sum()
        for col in channel_columns
    ]
})

channel_summary["Purchase_Share"] = (
    channel_summary["Total_Purchases"]
    / channel_summary["Total_Purchases"].sum()
    * 100
)

channel_summary["Avg_Purchases_Per_Buyer"] = (
    channel_summary["Total_Purchases"]
    / channel_summary["Channel_Buyers"].replace(0, np.nan)
)

display(
    channel_summary.sort_values(
        "Total_Purchases",
        ascending=False
    ).round(2)
)

,Channel,Total_Purchases,Channel_Buyers,Purchase_Share,Avg_Purchases_Per_Buyer
2,Store,12970,2225,46.18,5.83
0,Web,9150,2191,32.58,4.18
1,Catalog,5963,1654,21.23,3.61


In [14]:
top_channel = channel_summary.loc[
    channel_summary["Total_Purchases"].idxmax()
]

print("Top Channel:", top_channel["Channel"])

print(
    "Purchase Share:",
    round(top_channel["Purchase_Share"], 2),
    "%"
)

Top Channel: Store
Purchase Share: 46.18 %


In [15]:
campaign_columns = {
    "AcceptedCmp1": "Campaign 1",
    "AcceptedCmp2": "Campaign 2",
    "AcceptedCmp3": "Campaign 3",
    "AcceptedCmp4": "Campaign 4",
    "AcceptedCmp5": "Campaign 5",
    "Response": "Last Campaign"
}

campaign_summary = pd.DataFrame({
    "Campaign": list(campaign_columns.values()),
    "Accepted_Customers": [
        (df[col] == 1).sum()
        for col in campaign_columns
    ],
    "Recorded_Customers": [
        df[col].notna().sum()
        for col in campaign_columns
    ]
})

campaign_summary["Acceptance_Rate"] = (
    campaign_summary["Accepted_Customers"]
    / campaign_summary["Recorded_Customers"]
    * 100
)

campaign_summary = campaign_summary.sort_values(
    "Acceptance_Rate",
    ascending=False
)

display(campaign_summary.round(2))

,Campaign,Accepted_Customers,Recorded_Customers,Acceptance_Rate
5,Last Campaign,334,2240,14.91
3,Campaign 4,167,2240,7.46
2,Campaign 3,163,2240,7.28
4,Campaign 5,163,2240,7.28
0,Campaign 1,144,2240,6.43
1,Campaign 2,30,2240,1.34


In [16]:
top_campaign = campaign_summary.iloc[0]

print("Highest Acceptance Rate:", top_campaign["Campaign"])

print(
    "Acceptance Rate:",
    round(top_campaign["Acceptance_Rate"], 2),
    "%"
)

Highest Acceptance Rate: Last Campaign
Acceptance Rate: 14.91 %


In [17]:
rfm_summary = (
    df_rfm.groupby("RFM_Segment", dropna=False)
    .agg(
        Customers=("ID", "nunique"),
        Total_Spending=("Total_Spending", "sum"),
        Avg_Spending=("Total_Spending", "mean"),
        Avg_Purchases=("Total_Purchases", "mean"),
        Response_Rate=("Response", "mean")
    )
    .reset_index()
)

rfm_summary["Response_Rate"] *= 100

rfm_summary["Spending_Share"] = (
    rfm_summary["Total_Spending"]
    / rfm_summary["Total_Spending"].sum()
    * 100
)

display(
    rfm_summary.sort_values(
        "Total_Spending",
        ascending=False
    ).round(2)
)

,RFM_Segment,Customers,Total_Spending,Avg_Spending,Avg_Purchases,Response_Rate,Spending_Share
0,At Risk,402,458950,1141.67,19.41,30.10,33.82
1,Champions,329,417748,1269.75,20.50,15.50,30.78
3,Loyal Customers,230,235616,1024.42,19.28,16.52,17.36
2,Low Engagement,755,125945,166.81,6.99,13.38,9.28
5,Potential Loyalists,517,118509,229.22,7.40,4.45,8.73
4,Needs Review / No Recorded Purchase,7,220,31.43,0.14,0.00,0.02


In [18]:
at_risk = df_rfm[
    df_rfm["RFM_Segment"] == "At Risk"
]

print("At Risk Customers:", len(at_risk))

print(
    "At Risk Total Spending:",
    at_risk["Total_Spending"].sum()
)

At Risk Customers: 402
At Risk Total Spending: 458950


In [19]:
from scipy import stats

relationships = [
    ("Income", "Total_Spending"),
    ("Total_Purchases", "Total_Spending"),
    ("Recency", "Response")
]

statistical_results = []

for x, y in relationships:

    temp = df[[x, y]].dropna()

    corr, p_value = stats.spearmanr(
        temp[x],
        temp[y]
    )

    statistical_results.append({
        "Variable_1": x,
        "Variable_2": y,
        "Method": "Spearman",
        "Correlation": corr,
        "p_value": p_value,
        "Observations": len(temp)
    })

statistical_summary = pd.DataFrame(
    statistical_results
)

display(statistical_summary.round(4))

,Variable_1,Variable_2,Method,Correlation,p_value,Observations
0,Income,Total_Spending,Spearman,0.8509,0.0,2216
1,Total_Purchases,Total_Spending,Spearman,0.9083,0.0,2240
2,Recency,Response,Spearman,-0.1982,0.0,2240


In [20]:
dq_summary = pd.DataFrame({
    "Issue_ID": [
        "DQ-004",
        "DQ-005"
    ],
    "Issue": [
        "Deal Purchases exceed Total Purchases",
        "Zero Purchases with Positive Spending"
    ],
    "Affected_Customers": [
        int(df["Deal_Purchase_Issue"].sum()),
        int(df["Zero_Purchase_Spending_Issue"].sum())
    ],
    "Status": [
        "Needs Investigation",
        "Needs Investigation"
    ]
})

display(dq_summary)

,Issue_ID,Issue,Affected_Customers,Status
0,DQ-004,Deal Purchases exceed Total Purchases,3,Needs Investigation
1,DQ-005,Zero Purchases with Positive Spending,6,Needs Investigation


In [21]:
business_insights = pd.DataFrame([
    {
        "Area": "Customer Demographics",
        "Finding": "Review highest-value age and income groups",
        "Business_Question": "Which segments have higher spending?",
        "Recommendation": "Test segment-specific offers"
    },
    {
        "Area": "Product Analysis",
        "Finding": "Compare category spending and buyer penetration",
        "Business_Question": "Which product categories drive spending?",
        "Recommendation": "Review category-level marketing priorities"
    },
    {
        "Area": "Channel Analysis",
        "Finding": "Compare purchase counts by channel",
        "Business_Question": "Which channels are most frequently used?",
        "Recommendation": "Optimize channel-specific customer journeys"
    },
    {
        "Area": "Campaign Performance",
        "Finding": "Compare campaign acceptance rates",
        "Business_Question": "Which offers receive stronger responses?",
        "Recommendation": "Test effective offers across relevant segments"
    },
    {
        "Area": "RFM Segmentation",
        "Finding": "Identify Champions and At Risk customers",
        "Business_Question": "Which customers need retention or reactivation?",
        "Recommendation": "Develop segment-based CRM campaigns"
    },
    {
        "Area": "Data Quality",
        "Finding": "Investigate purchase consistency issues",
        "Business_Question": "Are purchase metrics reliable?",
        "Recommendation": "Validate metric definitions before reporting"
    }
])

display(business_insights)

,Area,Finding,Business_Question,Recommendation
0,Customer Demographics,Review highest-value age and income groups,Which segments have higher spending?,Test segment-specific offers
1,Product Analysis,Compare category spending and buyer penetration,Which product categories drive spending?,Review category-level marketing priorities
2,Channel Analysis,Compare purchase counts by channel,Which channels are most frequently used?,Optimize channel-specific customer journeys
3,Campaign Performance,Compare campaign acceptance rates,Which offers receive stronger responses?,Test effective offers across relevant segments
4,RFM Segmentation,Identify Champions and At Risk customers,Which customers need retention or reactivation?,Develop segment-based CRM campaigns
5,Data Quality,Investigate purchase consistency issues,Are purchase metrics reliable?,Validate metric definitions before reporting


In [22]:
output_dir = Path("../07_Documentation")

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

report_path = (
    output_dir / "Marketing_Analytics_Report.xlsx"
)

In [23]:
with pd.ExcelWriter(
    report_path,
    engine="openpyxl"
) as writer:

    executive_kpis.to_excel(
        writer,
        sheet_name="Executive_KPIs",
        index=False
    )

    age_summary.to_excel(
        writer,
        sheet_name="Age_Analysis",
        index=False
    )

    income_summary.to_excel(
        writer,
        sheet_name="Income_Analysis",
        index=False
    )

    product_summary.to_excel(
        writer,
        sheet_name="Product_Analysis",
        index=False
    )

    channel_summary.to_excel(
        writer,
        sheet_name="Channel_Analysis",
        index=False
    )

    campaign_summary.to_excel(
        writer,
        sheet_name="Campaign_Analysis",
        index=False
    )

    rfm_summary.to_excel(
        writer,
        sheet_name="RFM_Analysis",
        index=False
    )

    statistical_summary.to_excel(
        writer,
        sheet_name="Statistical_Analysis",
        index=False
    )

    dq_summary.to_excel(
        writer,
        sheet_name="Data_Quality",
        index=False
    )

    business_insights.to_excel(
        writer,
        sheet_name="Business_Insights",
        index=False
    )

print("Analytical Report Exported Successfully!")

Analytical Report Exported Successfully!
